# Lesson 55 · Completing the operators — giving the computation graph its joints: `__pow__` / `relu` / `tanh`, the full set of forward-pass operators

**Goal**: really complete the forward operators of `Value`: fit `__pow__`, `relu`, `tanh` and `exp` one by one, so the same computation graph can express a forward pass that looks much more like a neural network.

> Every operator has to leave behind one line of local derivative and one small diagram. Fit the joints first, and L56 can then send the gradient back along them.

🔗 **Aurora connection**: these operators correspond to every layer of a neural network — a linear transform (`+`, `*`, `**`) plus an activation function (`tanh`/`relu`) — and automatic differentiation is built on top of this set of operators. They will eventually land in the Aurora ML engine (`src/aurora/ml/`, a planned module).

← **Previous lesson** · [L54 · A `Value` computation graph](L54_value_autograd.ipynb)

> Last lesson covered **a `Value` computation graph**: scalar autodiff — forward values plus backward gradients, with hand-written add / mul nodes.
> This lesson fits the remaining joints and keeps extending what Value can express.

## Callback to the opening: what is L54's `Value` still missing?

L54 already spelled out the gradient rules for `__add__` / `__mul__`, but three kinds of "node moves" are still missing: `__pow__`, `relu`, and `tanh` / `exp`.
This lesson does just one thing: complete those operators, so the same computation graph can express a real neural network's forward pass.

## This lesson's story: how can a neural network learn "addition" and "nonlinearity" with the same code?

Imagine you're stacking building blocks: `z = w1*x1 + w2*x2 + b`, then `output = tanh(z)`. It looks like just two lines of maths, but training needs to know "if I change w1 by this much, how much does the loss change?" — every block needs its partial derivative.

Key insight: **every operator is a building block that knows how to pass the gradient back**.

```
Forward:  out = a * b           → out.data = a.data * b.data
Backward: ∂L/∂a = ∂L/∂out · b  → a.grad += out.grad * b.data
          ∂L/∂b = ∂L/∂out · a  → b.grad += out.grad * a.data
```

While each operator finishes its forward computation, it packs "how to pass the gradient backwards" into a **_backward closure** — and when `L.backward()` is called, we walk the DAG backwards from the output node, calling `_backward()` node by node.

**Three new operators**:

| Operator | Forward | Derivative (closed form) |
|---|---|---|
| `__pow__(n)` | `x^n` | `n·x^{n-1}` |
| `relu(x)` | `max(0,x)` | `1 if x>0 else 0` (a subgradient) |
| `tanh(x)` | `(e²ˣ-1)/(e²ˣ+1)` | `1 - tanh²(x)` (can be computed straight from the forward output) |

Task for this section: complete these three operators so that the perceptron `z=w1x1+w2x2+b`, `output=tanh(z)` backpropagates correctly from end to end.

In [ ]:
import math

# The complete Value class (with addition and multiplication; this section fills in the remaining operators)
class Value:
    def __init__(self, data, _children=(), _op=''):
        self.data = float(data)
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op

    def __repr__(self):
        return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"

    # --- existing operators ---
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        def _backward():
            self.grad  += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out

    def __radd__(self, other): return self + other

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        def _backward():
            self.grad  += other.data * out.grad
            other.grad += self.data  * out.grad
        out._backward = _backward
        return out

    def __rmul__(self, other): return self * other

    def __neg__(self):   return self * -1
    def __sub__(self, other): return self + (-other)
    def __rsub__(self, other): return Value(other) + (-self)
    def __truediv__(self, other):
        # Note: depends on __pow__. Implement __pow__ in the exercise cell below first, or this raises NotImplementedError
        return self * other**-1
    def __rtruediv__(self, other):
        # same as above: depends on __pow__
        return Value(other) * self**-1

    # --- operators still to implement (placeholders, replaced by the coding-task cell below) ---
    def __pow__(self, n): raise NotImplementedError
    def relu(self):       raise NotImplementedError
    def tanh(self):       raise NotImplementedError
    def exp(self):        raise NotImplementedError

    # --- backpropagation (topological order) ---
    def backward(self):
        topo, visited = [], set()
        def build(v):
            if v not in visited:
                visited.add(v)
                for c in v._prev: build(c)
                topo.append(v)
        build(self)
        self.grad = 1.0
        for v in reversed(topo):
            v._backward()

print("Value class defined")

## Digging deeper: what is the "topological sort" inside .backward()?

You may have noticed this code in the Value class:

```python
def backward(self):
    topo, visited = [], set()
    def build(v):
        if v not in visited:
            visited.add(v)
            for c in v._prev: build(c)  # recursively visit the child nodes
            topo.append(v)
    build(self)
    self.grad = 1.0
    for v in reversed(topo):
        v._backward()
```

"Topological sort" sounds complicated, but the core idea is actually very intuitive.

**Step 1: understand the "dependencies" in a computation graph**

Imagine building a house. First you lay the foundation, then you put up the walls, then fit the doors and windows, and finally decorate. The steps have an order — you can't fit the windows before the foundation is laid.

A computation graph is the same. For example:
```
x = Value(2.0)
y = Value(3.0)
z = x * y        # z depends on x and y
w = z + x        # w depends on z and x
```

Here `z` depends on `x` and `y`, and `w` depends on `z`. We say `z` is a "parent node" of `w`.

**Step 2: why walk backwards?**

Backpropagation pushes gradients from the "output" (`w`) back towards the "inputs" (`x`, `y`). But there is a rule: **a node can only pass its gradient on once every node that uses it (everything between it and the output) has finished sending its gradient back to it**.

The wrong way (starting from a leaf):
```
compute x.grad first  # x's gradient arrives from z and from w
  → needs z.grad, but it hasn't been computed yet!
```

The right way (reverse topological order):
```
1. Start with w.grad = 1 (the output)
2. Then compute z.grad (z's gradient comes from w)
3. Finally compute x.grad (x's gradient comes from z and from w)
```

**Step 3: the mechanics of the code**

```python
def build(v):
    if v not in visited:          # avoid visiting twice
        visited.add(v)
        for c in v._prev:         # visit all of v's inputs (child nodes)
            build(c)
        topo.append(v)            # v is only added after all its children have been handled
```

This recursion guarantees that **the closer a node is to the inputs, the earlier it is added to `topo`**. Flip it with `reversed()` and you get the order "from the output back to the inputs".

Example:
```
Computation graph: x → z → w
build(w):
  build(z):  (z not visited yet)
    build(x):(x not visited yet)
      (x has no children, so topo.append(x) right away)
    topo.append(z)
  topo.append(w)

Result: topo = [x, z, w]
reversed(topo) = [w, z, x]  ← exactly the order for backpropagation!
```

**What it means in practice**

This "topological sort" makes sure the gradients accumulate correctly. If the order is wrong (say, computing `x.grad` before `z.grad`), the gradient formulas go wrong.

## Concept 1: every operator = a new node + a _backward closure

Calling `a * b` makes two things happen:

1. **Forward**: `out = Value(a.data * b.data)`, with `_prev = {a, b}` attached
2. **Backward closure**: the chain rule is written into `_backward` — `a.grad += b.data * out.grad`

The closure captures the three live objects `a`, `b` and `out`. When `.backward()` is called they run in reverse topological order, and the gradients accumulate automatically into the leaf nodes' `.grad`.

### 🔍 Digging deeper: how does a closure "remember" variables?

If this is the first time you've seen code like this, you may wonder: why can the `_backward` function, defined inside `__mul__`, still reach the `self`, `other` and `out` from back then?

**An intuitive example**: compare two approaches

```python
# ❌ wrong way: store the number directly
def __mul__(self, other):
    out = Value(self.data * other.data)
    b_val = other.data  # store other's value "as it was then"
    def _backward():
        # ⚠️ problem: if other.grad gets modified later, this can't see it!
        self.grad += b_val * out.grad  # only knows the number; the object is lost
    return out

# ✅ right way: the closure captures object references
def __mul__(self, other):
    out = Value(self.data * other.data)
    def _backward():
        # ✅ the closure "remembers" the three objects self, other and out themselves
        # even when .backward() is called later, it sees the state of those objects at that time
        self.grad += other.data * out.grad
    out._backward = _backward
    return out
```

**Why do we need a closure (rather than storing the numbers directly)?**

In backpropagation, by the time `_backward()` is called, the gradient has already flowed into the downstream node through `out.grad`. Reading `other.data` at that point still gives the value "from the forward pass" (because after the forward pass the parameters don't change; only the gradients accumulate during the backward pass). The closure lets us postpone the action "read `other.data`" from "when it's defined" to "when it's called" — exactly what we need.

In other words:
- defining the closure → remember "these variables exist"
- calling the closure → read "the values those variables have now"

This is also the core mechanism of Python autodiff libraries such as PyTorch.

In [ ]:
# Demo: the _backward of an addition node
x = Value(3.0)
y = Value(4.0)
z = x + y       # z.data = 7.0
z.grad = 1.0
z._backward()   # trigger it by hand
print(f"x.grad = {x.grad}, y.grad = {y.grad}")  # both should be 1.0

## Extra: where does the power rule come from?

Before implementing `__pow__`, let's derive the formula `d(x^n)/dx = n·x^(n-1)` once, so you really understand and remember it (rather than learning it by rote).

**Case 1: an integer power (the simplest)**

For `f(x) = x³`, use the definition of the derivative:
```
f'(x) = lim[h→0] (f(x+h) - f(x)) / h
      = lim[h→0] ((x+h)³ - x³) / h
```

Expand `(x+h)³` with the binomial theorem:
```
(x+h)³ = x³ + 3x²h + 3xh² + h³
```

Substitute:
```
f'(x) = lim[h→0] (x³ + 3x²h + 3xh² + h³ - x³) / h
      = lim[h→0] (3x²h + 3xh² + h³) / h
      = lim[h→0] (3x² + 3xh + h²)    ← cancel the h
      = 3x²   ← as h→0, both 3xh and h² vanish
```

Spot the pattern: the derivative of `x^3` is `3x²` (the coefficient is the old exponent 3, and the new exponent is 3-1=2).

**Generalizing to any integer power n**

With a similar method (or with the binomial theorem), you can prove:
```
d(x^n)/dx = n·x^(n-1)
```

Check a few examples:
- `x¹` → derivative `1·x^0 = 1`  ✓ (a constant has derivative 0, a linear function has derivative 1)
- `x²` → derivative `2·x^1 = 2x`  ✓
- `x^(-1) = 1/x` → derivative `-1·x^(-2) = -1/x²`  ✓ (matches the quotient rule)

**Why does it work for negative powers too?**

Although the derivation used the binomial expansion, the formula also holds for negative integers. For example `x^(-2)`:
```
d(x^(-2))/dx = -2·x^(-3) = -2/x³
```

This agrees with differentiating `1/x²` using the quotient rule.

**How this lesson uses it**

In the `_backward` of `__pow__`, we use exactly this formula:
```python
self.grad += n * (self.data ** (n-1)) * out.grad
```

Here `n·self.data^(n-1)` is the "local derivative"; multiply it by `out.grad` (the upstream gradient) and you get this step's contribution to the input.

## Extra: how is the tanh derivative `1 - tanh²(x)` derived?

Before we see "why the tanh gradient can be computed straight from the forward output", let's derive the formula `d(tanh(x))/dx = 1 - tanh²(x)` once.

**Step 1: recall the definition of tanh**

```
tanh(x) = (e^x - e^(-x)) / (e^x + e^(-x))
        = (e^(2x) - 1) / (e^(2x) + 1)    ← two equivalent forms; we use the second
```

**Step 2: differentiate with the quotient rule**

Recall the quotient rule: `d/dx [f/g] = (f'g - fg') / g²`

Here:
- `f = e^(2x) - 1` → `f' = 2e^(2x)`
- `g = e^(2x) + 1` → `g' = 2e^(2x)`

Substitute:
```
d(tanh)/dx = (2e^(2x) · (e^(2x) + 1) - (e^(2x) - 1) · 2e^(2x)) / (e^(2x) + 1)²
```

Expand the numerator:
```
= (2e^(2x) · e^(2x) + 2e^(2x) - 2e^(2x) · e^(2x) + 2e^(2x)) / (e^(2x) + 1)²
= (2e^(2x) + 2e^(2x)) / (e^(2x) + 1)²
= 4e^(2x) / (e^(2x) + 1)²
```

**Step 3: re-express it in terms of the value of tanh**

Now let `t = tanh(x) = (e^(2x) - 1) / (e^(2x) + 1)`.

From the definition `t = (e^(2x) - 1) / (e^(2x) + 1)` we can derive:
```
e^(2x) - 1 = t(e^(2x) + 1)
e^(2x) - 1 = t·e^(2x) + t
e^(2x) - t·e^(2x) = 1 + t
e^(2x)(1 - t) = 1 + t
e^(2x) = (1 + t) / (1 - t)
```

We could also try starting from `e^(2x) + 1 = (e^(2x) - 1 + 2) = t(e^(2x) + 1) + 2`... but there's actually a neater way:

We know `e^(2x) + 1 = e^(2x) + 1`; substitute `e^(2x) = (1+t)/(1-t)` from above:
```
e^(2x) + 1 = (1+t)/(1-t) + 1 = (1+t + 1-t)/(1-t) = 2/(1-t)
```

So:
```
d(tanh)/dx = 4e^(2x) / (e^(2x) + 1)²
           = 4 · (1+t)/(1-t) / (2/(1-t))²
           = 4 · (1+t)/(1-t) · (1-t)²/4
           = (1+t)(1-t)
           = 1 - t²
```

**Conclusion**

```
d(tanh(x))/dx = 1 - tanh²(x)
```

What does this tell us? The derivative depends only on the value of `tanh(x)` itself — you don't need to know what the original input `x` was! That's why the code can simply write `(1 - out.data**2) * out.grad`.

## Concept 2: the tanh gradient is expressed directly through the forward output

`tanh(x) = (e^{2x} - 1) / (e^{2x} + 1)`, and its derivative is:

```
d(tanh(x))/dx = 1 - tanh(x)^2
```

Implementation trick: `out.data` is the `tanh(x)` value we've already computed, so the backward pass can just write:

```python
self.grad += (1 - out.data**2) * out.grad
```

**Why can we use only the forward output, and not the original input x?**

This is a mathematical property of tanh: its derivative can be expressed in terms of **the function's own value**.

Concretely:
- in the forward pass we computed `out.data = tanh(x)` and stored it in node `out`
- in the backward pass we need the gradient `d(tanh(x))/dx = 1 - tanh(x)²`
- the key: we don't need the actual value of `x`! We only need the value of `tanh(x)`
- substitute `out.data` for `tanh(x)`: `1 - out.data²` is the value of the derivative

The benefits:
1. Less computation: no need to call `math.tanh(x)` again
2. Less storage: the backward closure doesn't have to keep `x` specially; `out.data` is enough
3. Generality: for any activation function `f`, as long as its derivative `f'` can be expressed through `f(x)`, the same trick works

Other activation functions may not be so lucky. `sigmoid(x) = 1/(1+e^(-x))` happens to be: its derivative is `sigmoid(x)·(1-sigmoid(x))`, which can also be computed from the forward value. But some functions can't — for example the derivative of `elu(x) = x if x>0 else alpha·(e^x-1)` needs to know whether `x` is positive or negative, and you can't tell that from the output value alone.

In [ ]:
# Check the formula: numerical difference vs analytic gradient
import math

x_val = 0.8
h = 1e-5
analytic = 1 - math.tanh(x_val)**2
numeric  = (math.tanh(x_val + h) - math.tanh(x_val - h)) / (2*h)
print(f"Analytic gradient: {analytic:.6f}")
print(f"Numerical gradient: {numeric:.6f}")
print(f"Error: {abs(analytic - numeric):.2e}")

## Concept 3: the ReLU gradient and the "subgradient"

`relu(x) = max(0, x)`: for `x > 0` the gradient is 1, for `x < 0` it is 0, and at `x = 0` we take the subgradient 0:

```
d(relu(x))/dx = 1  if x > 0 else 0
```

Backward implementation:

```python
self.grad += (out.data > 0) * out.grad
```

`out.data > 0` is a boolean, which in Python equals `1` or `0`, so you can multiply by it directly.

**ReLU is "not differentiable" at x=0 — so why do we insist on choosing a gradient there?**

ReLU has a "kink" at `x=0` — the graph switches abruptly from the lower line (`y=0`) to the upper line (`y=x`). That means:
- approaching 0 from the left: the gradient is 0 (on `x<0` the output is always 0)
- approaching 0 from the right: the gradient is 1 (on `x>0` the output is `x` itself)

In more advanced maths this is called "non-smooth", and the ordinary "derivative" isn't defined at that point. But we introduce the idea of a "**subgradient**": for a non-smooth function like this, at the kink the gradient may take any value between the left and right gradients.

**Why choose 0 in practice (rather than 1 or something else)?**

The simple reasons:
1. **Probability**: in real data, the chance that a neuron's input is exactly 0 is close to zero (it almost never happens)
2. **Stability**: 0 is the conservative choice — no gradient is passed, so that input is "cut off"; choosing 1 would match the behaviour on the positive side; the difference between the two is negligible
3. **Simpler code**: the code is just `(out.data > 0) * out.grad`, which automatically gives 0 when `out.data == 0` — very neat

So in our implementation we choose 0: it doesn't affect training (because exactly 0 is so rare), and it keeps the code short and clear.

**Extra: the dead ReLU problem**

The downside of ReLU is that on `x < 0` the gradient is always 0, which can leave some neurons never activated (no gradient ever flows to them). This is called the "dead ReLU" problem. Leaky ReLU and ELU are fixes for it:
- Leaky ReLU: for `x<0` the gradient is a small positive number (e.g. 0.01) instead of 0
- ELU: on the negative side it uses a smooth exponential, avoiding the hard cut-off

In [ ]:
# Demo of the ReLU subgradient
vals = [-2.0, 0.0, 3.0]
for v in vals:
    grad = float(v > 0)
    print(f"relu'({v:+.1f}) = {grad}")

## 1. ✏️ Implement `Value.__pow__`, `Value.relu`, `Value.tanh`, `Value.exp`

**A three-step implementation template** (every operator has the same structure):

```python
def __pow__(self, n):
    out = Value(self.data ** n, _children=(self,), _op=f'**{n}')
    def _backward():
        self.grad += n * (self.data ** (n-1)) * out.grad
    out._backward = _backward
    return out
```

**The _backward formulas of the four operators**:

| Operator | Forward | Backward gradient formula | Notes |
|---|---|---|---|
| `__pow__(n)` | `x^n` | `n · x^(n-1) · out.grad` | power rule |
| `relu` | `max(0, x)` | `(x > 0) · out.grad` | subgradient (0 at 0) |
| `tanh` | `tanh(x)` | `(1 - tanh²(x)) · out.grad` | computed straight from the forward output |
| `exp` | `e^x` | `e^x · out.grad` | its derivative is itself |

**What exp() is for**

Although `exp` isn't used directly in this lesson (we use the library function `math.tanh`), we implement it for completeness and for later extensions. In some situations you may want to build tanh yourself, starting from the exponential function:
```
tanh(x) = (e^(2x) - 1) / (e^(2x) + 1)
        = 2*e^(2x) / (e^(2x) + 1) - 1
```
That's when you need the `exp` operator. Loss functions such as softmax and cross-entropy also use exp in their backward pass.

**Acceptance criteria**:
- `Value(-2).__pow__(2).data == 4.0`
- `Value(-1.0).relu().data == 0.0`, `Value(2.0).relu().data == 2.0`
- after the perceptron's `backward()`, the error against numerical differences is < 1e-5
- `Value(1.0).exp().data` is about 2.718 (Euler's number e)

In [ ]:
# ✏️ TODO: implement all four operators below, replacing the placeholder raise NotImplementedError in the Value class

def __pow__(self, n):
    # ✏️ TODO
    assert isinstance(n, (int, float))
    t   = ...                        # ✏️ TODO: self.data**n
    out = Value(t, (self,), f'**{n}')  # _prev and _op are already filled in; you only need t
    def _backward():
        self.grad += ...    # ✏️ TODO: n * self.data**(n-1) * out.grad
    out._backward = _backward
    return out

def relu(self):
    t = ...                 # ✏️ TODO: max(0, self.data)
    out = Value(t, (self,), 'relu')
    def _backward():
        self.grad += ...    # ✏️ TODO: (t > 0) * out.grad
    out._backward = _backward
    return out

def tanh(self):
    t = ...                 # ✏️ TODO: math.tanh(self.data)
    out = Value(t, (self,), 'tanh')
    def _backward():
        self.grad += ...    # ✏️ TODO: (1 - t**2) * out.grad
    out._backward = _backward
    return out

def exp(self):
    t = ...                 # ✏️ TODO: math.exp(self.data)
    out = Value(t, (self,), 'exp')
    def _backward():
        self.grad += ...    # ✏️ TODO: t * out.grad
    out._backward = _backward
    return out

# attach them to the Value class
Value.__pow__ = __pow__
Value.relu    = relu
Value.tanh    = tanh
Value.exp     = exp

In [ ]:
# Check: basic forward values
assert Value(-2.0).__pow__(2).data == 4.0,   "__pow__ forward is wrong"
assert Value(-2.0).relu().data == 0.0,        "relu forward: a negative input should give 0"
assert Value( 0.5).relu().data == 0.5,        "relu forward: a positive input should pass straight through"
assert abs(Value(1.0).tanh().data - 0.7616) < 1e-4, "tanh(1.0) should be ≈ 0.7616"
assert abs(Value(1.0).exp().data  - math.e)  < 1e-5, "exp(1.0) should be ≈ e"

# Check: __pow__ backward
a = Value(3.0)
b = a**2          # b = 9, db/da = 2*3 = 6
b.backward()
assert abs(a.grad - 6.0) < 1e-9, f"__pow__ backward gradient should be 6.0, got {a.grad}"

# Check: relu backward (positive side)
a = Value(2.0)
r = a.relu()
r.backward()
assert a.grad == 1.0, "relu backward gradient on the positive side should be 1.0"

# Check: relu backward (negative side)
a = Value(-1.0)
r = a.relu()
r.backward()
assert a.grad == 0.0, "relu backward gradient on the negative side should be 0.0"

# Check: tanh backward
a = Value(0.0)
t = a.tanh()       # tanh(0)=0, gradient=1-0^2=1
t.backward()
assert abs(a.grad - 1.0) < 1e-9, "the backward gradient of tanh(0) should be 1.0"

print("✅ All operator forward & backward checks passed")

## Parameter experiment: a 2-input perceptron's forward pass and a check of the chain rule

Build a minimal perceptron with the `Value` operators:

```
z = w1*x1 + w2*x2 + b
output = tanh(z)         # activation function
```

Parameters: `w1=0.5, w2=-0.3, b=0.1, x1=1.0, x2=2.0`

**What to expect**:
- `z.data` should be `0.5*1 + (-0.3)*2 + 0.1 = 0.0`
- `output.data` should be `tanh(0.0) = 0.0`
- after calling `output.backward()`, the gradients flow back to the parameters by the chain rule

**Working through the chain rule**

In this example, let's derive the gradient of `w1` by hand. Let loss = `output` (to keep things simple, assume `output` itself is the value we care about).

**A quick aside: why does the derivation below have only two factors, when the standard chain rule has three?**

If you've seen the chain rule elsewhere, you may have seen the "full" version:

```
∂L/∂w = (∂L/∂output) · (∂output/∂z) · (∂z/∂w)
```

Here `L` is the "loss function" — the number we actually want to minimize during training. The three factors correspond to three consecutive hops on the computation graph: `w changes → z changes with it → output changes with it → L changes with it`; multiply the "sensitivity" of each hop together and you get the total sensitivity of `L` to `w`.

An analogy: you want to know "how your pay rise affects your mood tonight", and the chain is "pay rise → savings this month → mood". But if your mood tonight is simply you staring at your savings balance, with no extra mental processing (no "comparing yourself to others" step in between), then the factor "effect of savings on mood" is 1 — one more dollar in savings means exactly one more dollar of mood (measured in savings), no more, no less, because your mood **is** the savings number.

Back to the formula: in this lesson's perceptron example we haven't introduced a loss function yet — we simply set `loss = output`, meaning "do nothing extra to output; treat it directly as the quantity we care about". Mathematically this is the **identity function**: `L(output) = output`. What is the derivative of the identity function? Recall the line `y = x`: its slope is always 1, so:

```
∂L/∂output = ∂(output)/∂output = 1
```

Put that 1 back into the three-factor chain rule:

```
∂L/∂w = 1 · (∂output/∂z) · (∂z/∂w) = (∂output/∂z) · (∂z/∂w)
```

— that's why the derivation below is written with just two factors: the first factor wasn't forgotten; it simply comes out to exactly 1, and multiplying by 1 changes nothing, so it "disappears".

> **Reminder**: this is only a simplification for this lesson's example! As soon as later lessons plug in a real loss function (such as mean squared error, MSE, or cross-entropy), `∂L/∂output` is no longer 1, and all three factors of the chain rule are needed again — you'll have to compute that factor first and multiply it back in. L56 covers this step.

Along the path through the computation graph:
```
w1 --→ (w1*x1) --→ (w1*x1 + w2*x2 + b) --→ tanh(...) --→ output
                   (i.e. z)                  (i.e. output)
```

The chain rule says: `∂output/∂w1 = (∂output/∂z) · (∂z/∂w1)` (here `output` is `L`, so the factor `∂L/∂output`, which is always 1, is left out)

Broken down:
- `∂output/∂z`: output = tanh(z), so `∂output/∂z = tanh'(z) = 1 - tanh²(z) = 1 - 0² = 1`
- `∂z/∂w1`: z = w1*x1 + ..., so `∂z/∂w1 = x1 = 1`

Therefore: `∂output/∂w1 = 1 · 1 = 1`

Likewise, the gradient of `w2`:
- `∂output/∂w2 = (∂output/∂z) · (∂z/∂w2) = tanh'(z) · x2 = 1 · 2 = 2`
- the gradient of `b`: `∂output/∂b = tanh'(z) · 1 = 1`

**Key observations**:
1. The gradient flows backwards from the "output" to the "inputs"
2. Each step multiplies by a "local gradient" (the derivative of that step)
3. The `.backward()` in the code walks the computation graph backwards by exactly this rule, accumulating gradients automatically

**Compare with a relu activation**

If you swap `tanh` for `relu`, then at `z=0` we have `relu'(0) = 0` (our subgradient choice), so every gradient becomes 0 — the path is "cut". This is the dead ReLU effect.

In [ ]:
# 2-input perceptron: forward + backward
w1 = Value(0.5)
w2 = Value(-0.3)
b  = Value(0.1)
x1 = Value(1.0)
x2 = Value(2.0)

z      = w1*x1 + w2*x2 + b   # linear combination
output = z.tanh()              # tanh activation

print(f"z.data     = {z.data:.4f}      (expected 0.0)")
print(f"output.data= {output.data:.4f}  (expected tanh(0.0) = 0.0)")

output.backward()
print(f"\nAfter backpropagation:")
print(f"w1.grad = {w1.grad:.4f}   (= x1 * tanh'(z) = 1.0 * 1.0)")
print(f"w2.grad = {w2.grad:.4f}   (= x2 * tanh'(z) = 2.0 * 1.0)")
print(f"b.grad  = {b.grad:.4f}   (= tanh'(z) = 1.0)")

# Compare with a relu activation (fresh, independent inputs, so the gradients don't pile onto the tanh demo above)
w1r, w2r, br = Value(0.5), Value(-0.3), Value(0.1)
x1r, x2r = Value(1.0), Value(2.0)   # ← fresh ones; x1/x2 from above are not reused
zr = w1r*x1r + w2r*x2r + br
out_r = zr.relu()
out_r.backward()
print(f"\nrelu activation: output={out_r.data:.4f}, w1.grad={w1r.grad:.4f}, w2.grad={w2r.grad:.4f}")
print("(relu's subgradient at z=0 is 0, so the gradient flow is cut off)")

## Wrap-up

This section implemented four operators, `__pow__`, `relu`, `tanh` and `exp`; each one returns a new `Value` node and packs a backward-gradient closure along with it. `Value` now doesn't just do arithmetic, it has also started "keeping the books": `out.data` holds the forward value, and `out.grad` holds the gradient flowing back. With the complete set of operators, `Value` can describe a computation graph of any depth — matching the forward computation of every layer in the Aurora ML engine (`src/aurora/ml/`, a planned module). The next lesson (**L56**) takes `.backward()` apart and works it through by hand: topologically sorting a whole multi-layer computation graph and checking, node by node, the chain-rule propagation "gradient = local gradient × upstream gradient".

## ✏️ Whiteboard challenge: operator gradients by hand (target: 10 minutes)

Cover the screen and derive on paper:

**Question 1**: what is the gradient of `z = x**3` at `x=2`?
(derivative: 3x² → 3×4 = 12)

**Question 2**: why can the derivative of `tanh` be expressed through the forward output?
(`d/dx tanh(x) = 1 - tanh²(x)`; you don't need the original x, only `out.data`)

**Question 3**: for the computation graph `L = relu(x*w + b)` with `x=1, w=2, b=-1`:
- forward values: `z = x*w+b = ?`, `L = relu(z) = ?`
- backward: `∂L/∂w = ?` (remember relu's gradient is 1 when z>0)

**Question 4**: why does ReLU use the subgradient 0 at `x=0` (rather than leaving it undefined)?
(in practice, landing exactly on 0 has probability zero, and using 0 doesn't hurt training stability)

**Question 5**: `c = a**2 + b**2` with `a=3, b=4`: after `c.backward()`, what are `a.grad` and `b.grad`?
(`∂c/∂a = 2a = 6`, `∂c/∂b = 2b = 8`)

When you've finished the derivations, run the cell below to check your answers.

In [ ]:
# ✏️ Answer-check cell
import math

# Note: while the operators aren't implemented, the `...` in the TODOs makes Value(...) raise TypeError,
# and the placeholder methods raise NotImplementedError — both count as "not implemented yet", and the written answer is shown instead.

# Q1: z=x**3, x=2 → grad=12
try:
    x = Value(2.0)
    z = x.__pow__(3)
    z.grad = 1.0
    z._backward()
    assert abs(x.grad - 12.0) < 1e-12, f"x.grad={x.grad}"
    print(f"Q1 ✅  d/dx(x³)|x=2 = {x.grad:.1f} = 3×2² = 12")
except (NotImplementedError, TypeError):
    print(f"Q1 ✅  d/dx(x³)|x=2 = 3×2²=12 (__pow__ not implemented yet)")

# Q2: the tanh derivative from the forward output
x_val = 0.8
tanh_val = math.tanh(x_val)
grad_analytic = 1 - tanh_val**2
grad_numeric  = (math.tanh(x_val+1e-5) - math.tanh(x_val-1e-5)) / (2e-5)
assert abs(grad_analytic - grad_numeric) < 1e-9
print(f"Q2 ✅  tanh'({x_val})=1-tanh²={grad_analytic:.6f}, numerical difference={grad_numeric:.6f}")

# Q3: relu(x*w+b), x=1, w=2, b=-1
try:
    x, w, b = Value(1.0), Value(2.0), Value(-1.0)
    z = x*w + b       # z.data = 1.0
    L = z.relu()      # L.data = 1.0 (z>0)
    L.backward()
    assert abs(L.data - 1.0) < 1e-12
    assert abs(w.grad - 1.0) < 1e-12, f"w.grad={w.grad}"
    print(f"Q3 ✅  z=x*w+b={z.data:.1f}>0, L=relu(z)={L.data:.1f}, ∂L/∂w={w.grad:.1f}=x=1")
except (NotImplementedError, TypeError):
    print(f"Q3 ✅  x=1,w=2,b=-1: z=1>0, L=relu(z)=1, ∂L/∂w=x=1 (relu not implemented yet)")

# Q4: the relu subgradient (a concept check)
relu_grads = {-1.0: 0, 0.0: 0, 1.0: 1}
for v, expected in relu_grads.items():
    got = float(v > 0)
    assert got == expected
print(f"Q4 ✅  relu subgradient: x<0→0, x=0→0, x>0→1 (a subgradient, safe in practice)")

# Q5: c=a²+b², a=3, b=4
try:
    a, b = Value(3.0), Value(4.0)
    c = a**2 + b**2
    c.backward()
    assert abs(a.grad - 6.0) < 1e-12, f"a.grad={a.grad}"
    assert abs(b.grad - 8.0) < 1e-12, f"b.grad={b.grad}"
    print(f"Q5 ✅  c=a²+b²: ∂c/∂a={a.grad:.1f}=2a=6, ∂c/∂b={b.grad:.1f}=2b=8")
except (NotImplementedError, TypeError):
    print(f"Q5 ✅  c=a²+b²: ∂c/∂a=2×3=6, ∂c/∂b=2×4=8 (operators not implemented yet)")
print("\n🎉 Operator-gradient whiteboard challenge passed!")

In [ ]:
# ✏️ Self-review for this lesson
l55_review = {
    "pow_backward":            None,  # remember d(x^n)/dx=n·x^(n-1) and implemented _backward correctly? True/False
    "relu_subgradient":        None,  # understand why relu uses the subgradient 0 at x=0? True/False
    "tanh_forward_trick":      None,  # remember tanh'=1-tanh² and compute it straight from out.data? True/False
    "all_ops_implemented":     None,  # __pow__/relu/tanh/exp all implemented and the perceptron backward passes? True/False
    "whiteboard_passed":       None,  # all 5 whiteboard-challenge derivations done by hand? True/False
}

unfilled = [k for k, v in l55_review.items() if v is None]
assert not unfilled, f'not filled in yet: {unfilled}'
weak = [k for k, v in l55_review.items() if v is False]
if weak:
    print(f'⚠️  needs more work: {weak}')
else:
    print('✅ L55 fully cleared! On to L56: backpropagation by hand')

---

→ **Next lesson** · [L56 · Backpropagation by hand](L56_backward_pass.ipynb)

> Next lesson covers **backpropagation by hand**: unrolling the chain rule layer by layer — gradient = local gradient × upstream gradient.